# Deep dive 1/13: `customers`

Checks whether `silver.dim_customers` correctly implements every transformation `bronze.customers` actually needs -- not just "does it run," but "is each column typed, cleaned, and validated the way the data actually requires."

Source of truth for what Silver *should* do: `data_pipelines/silver/table_specs.py`'s `customers_spec` and `data_pipelines/silver/transforms.py`. Source of truth for what the data *actually* looks like: this notebook, run against your real `data/latam_bank.duckdb`.

Run top to bottom. Section 10 is a checklist to fill in with what you found; free cells are at the end for anything this doesn't already cover.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 1. Column inventory -- does `silver.dim_customers` have everything `customers_spec` says it should?

This isn't "does the table exist" (you already know it does from `run_silver.py`'s output) -- it's "does every column `table_specs.py` declares actually show up with the right name," which catches a spec/build drift that a successful run wouldn't otherwise surface.

In [ ]:
# Exact column list customers_spec declares (target names, i.e. what should land in Silver),
# transcribed from data_pipelines/silver/table_specs.py's customers_spec -- keep this in sync if
# that spec changes.
expected_silver_columns = [
    "customer_id", "document_number", "document_type", "first_name", "last_name",
    "date_of_birth", "gender", "email", "mobile_phone", "landline_phone", "address",
    "city", "state", "country", "postal_code", "detected_accent", "segment",
    "credit_score", "estimated_monthly_income", "occupation", "marital_status",
    "education_level", "registration_date", "registration_branch_id", "customer_status",
    "last_updated", "accepts_marketing",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_customers'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_customers (spec declares, table lacks):", missing or "none")
print("Extra in silver.dim_customers (table has, spec doesn't declare):", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

In [ ]:
# Bronze's raw columns, for reference -- everything below this line should be accounted for by
# either a Silver column or a deliberate drop (Hive partition columns, _ingested_at, etc.)
bronze_cols = q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='customers'
    ORDER BY ordinal_position
""")
bronze_cols

In [ ]:
# Silver columns
bronze_cols = q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_customers'
    ORDER BY ordinal_position
""")
bronze_cols

## 2. Row-count reconciliation -- did dedup do what it's supposed to?

`silver.dim_customers` should have exactly one row per distinct `customer_id`, keeping the latest `_ingested_at`. This is the same invariant the quality gate's `silver_row_delta_unexplained` check enforces, reproduced here directly against your data.

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.customers")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT customer_id) AS n FROM bronze.customers")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_customers")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT customer_id) AS n FROM silver.dim_customers")["n"][0]

print(f"bronze.customers:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct customer_id")
print(f"silver.dim_customers:  {silver_total:,} rows, {silver_distinct_pk:,} distinct customer_id")
print()
print("Duplicate raw rows in Bronze (resends/reloads):", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- dedup did not produce one row per customer_id, investigate")

In [ ]:
# Any duplicate customer_id left in Silver itself? Should always be empty.
q("""
    SELECT customer_id, count(*) AS n
    FROM silver.dim_customers
    GROUP BY 1 HAVING count(*) > 1
""")

## 3. NOT NULL audit -- does Bronze actually satisfy the data dictionary's constraints?

The data dictionary marks these columns `NOT NULL`: `customer_id`, `document_number`, `document_type`, `first_name`, `last_name`, `date_of_birth`, `city`, `state`, `country`, `segment`, `registration_date`, `registration_branch_id`, `customer_status`, `last_updated`, `accepts_marketing`. Silver doesn't enforce this (DuckDB tables here carry no constraints), so if any of these are actually nullable in the real data, Silver will silently carry the NULLs through -- worth knowing before a report assumes these are always populated.

In [ ]:
declared_not_null = [
    "customer_id", "document_number", "document_type", "first_name", "last_name",
    "date_of_birth", "city", "state", "country", "segment", "registration_date",
    "registration_branch_id", "customer_status", "last_updated", "accepts_marketing",
]

rows = []
total = q("SELECT count(*) AS n FROM bronze.customers")["n"][0]
for col in declared_not_null:
    nulls = q(f"""
        SELECT count(*) AS n FROM bronze.customers
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "actual_nulls": nulls,
                 "null_rate": round(nulls / total, 4) if total else None})

null_audit = pd.DataFrame(rows)
null_audit

## 4. Domain / categorical checks

Compares actual distinct values against what the data dictionary declares, for every categorical-ish column. A value outside the declared set isn't necessarily wrong (the dictionary itself admits synthetic imperfections), but Silver doesn't flag or filter any of these except `customer_status` (already checked by the quality gate) -- worth seeing what the others actually contain.

In [ ]:
# document_type -- dictionary: DNI, CURP, CC, CE, Passport
q("SELECT document_type, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC")

In [ ]:
# gender -- dictionary: M, F, O
q("SELECT gender, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC")

In [ ]:
# segment -- dictionary: Premium, Plus, Basic, Student
q("SELECT segment, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC")

In [ ]:
# customer_status -- dictionary: Active, Inactive, Suspended, Closed
# NOTE: table_specs.py/quality's domain check actually uses {Active, Inactive, Blocked, Closed} --
# "Suspended" (dictionary) vs "Blocked" (contract) is worth reconciling; this cell shows the raw
# truth either way. This is the known warning from the quality report (4,407/150,000).
q("SELECT customer_status, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC")

In [ ]:
# marital_status / education_level -- not declared in the dictionary's domain list at all,
# so there's no "correct" set to check against -- just seeing what's actually there.
print("marital_status:")
display(q("SELECT marital_status, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC"))
print("education_level:")
display(q("SELECT education_level, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC"))

In [ ]:
# marital_status / education_level -- not declared in the dictionary's domain list at all,
# so there's no "correct" set to check against -- just seeing what's actually there.
print("marital_status:")
display(q("SELECT marital_status, count(*) AS n FROM silver.dim_customers GROUP BY 1 ORDER BY n DESC"))
print("education_level:")
display(q("SELECT education_level, count(*) AS n FROM silver.dim_customers GROUP BY 1 ORDER BY n DESC"))

Users note: are these 'NaN' values in marital_status and education_level acctually 'NaN' string values or are they null?

## 5. Numeric range sanity

`credit_score` and `estimated_monthly_income` are cast to DOUBLE in Silver but never range-checked. The dictionary states `credit_score` should be 300-850 -- worth confirming the real data actually stays inside that band, and that income has no negative/absurd outliers Silver would otherwise pass through untouched.

In [ ]:
q("""
    SELECT
        min(TRY_CAST(credit_score AS DOUBLE)) AS min_credit_score,
        max(TRY_CAST(credit_score AS DOUBLE)) AS max_credit_score,
        count(*) FILTER (WHERE TRY_CAST(credit_score AS DOUBLE) NOT BETWEEN 300 AND 850) AS out_of_declared_range,
        count(*) FILTER (WHERE credit_score IS NULL) AS original_nulls,  -- users note
        count(*) FILTER (WHERE trim(credit_score) = '') AS empty_strings_nulls,  -- users note
        count(*) FILTER (WHERE credit_score IS NOT NULL and TRY_CAST(credit_score AS DOUBLE) IS NULL) AS casting_fail_nulls  -- users note
    FROM bronze.customers
""")

In [ ]:
q("""
    SELECT
        typeof(credit_score), -- users note: check if data type is casted in silver
        min(credit_score) AS min_credit_score,
        max(credit_score) AS max_credit_score,
        count(*) FILTER (WHERE credit_score NOT BETWEEN 300 AND 850) AS out_of_declared_range,
        count(*) FILTER (WHERE credit_score IS NULL) AS nulls
    FROM silver.dim_customers
    GROUP BY ALL
""")

In [ ]:
q("""
    SELECT
        min(TRY_CAST(estimated_monthly_income AS DOUBLE)) AS min_income,
        max(TRY_CAST(estimated_monthly_income AS DOUBLE)) AS max_income,
        count(*) FILTER (WHERE TRY_CAST(estimated_monthly_income AS DOUBLE) < 0) AS negative_income_rows,
        count(*) FILTER (WHERE estimated_monthly_income IS NULL) AS original_nulls,  -- users note
        count(*) FILTER (WHERE trim(estimated_monthly_income) = '') AS empty_strings_nulls,  -- users note
        count(*) FILTER (WHERE estimated_monthly_income IS NOT NULL and TRY_CAST(estimated_monthly_income AS DOUBLE) IS NULL) AS casting_fail_nulls  -- users note
    FROM bronze.customers
""")

In [ ]:
q("""
    SELECT
        typeof(estimated_monthly_income), -- users note: check if data type is casted in silver
        min(TRY_CAST(estimated_monthly_income AS DOUBLE)) AS min_income,
        max(TRY_CAST(estimated_monthly_income AS DOUBLE)) AS max_income,
        count(*) FILTER (WHERE TRY_CAST(estimated_monthly_income AS DOUBLE) < 0) AS negative_income_rows,
        count(*) FILTER (WHERE estimated_monthly_income IS NULL) AS nulls
    FROM silver.dim_customers
    GROUP BY ALL
""")

## 6. Date sanity

Three checks Silver's plain `TRY_CAST(... AS DATE/TIMESTAMP)` doesn't do: implausible ages from `date_of_birth`, `registration_date` falling outside the dataset's stated window (June 2023 - June 2026 per the data dictionary), and the `last_updated` anomaly the spec's own docstring already flags (a 2027 value against a dataset that otherwise runs through 2026-06-17).

In [ ]:
# Age sanity from date_of_birth -- flags anyone implausibly old (>100) or with a birth date in
# the future/inside the dataset window (implying a data entry error, not a real customer).
q("""
    SELECT
        min(TRY_CAST(date_of_birth AS DATE)) AS earliest_dob,
        max(TRY_CAST(date_of_birth AS DATE)) AS latest_dob,
        count(*) FILTER (WHERE date_diff('year', TRY_CAST(date_of_birth AS DATE), current_date) > 100) AS implausibly_old,
        count(*) FILTER (WHERE TRY_CAST(date_of_birth AS DATE) > current_date) AS future_birth_dates,
        count(*) FILTER (WHERE date_of_birth IS NULL OR trim(date_of_birth) = '' or TRY_CAST(date_of_birth AS DATE) is null) AS nulls
    FROM bronze.customers
""")

In [ ]:
# registration_date vs the dataset's stated window (2023-06-17 to 2026-06-17)
q("""
    SELECT
        min(TRY_CAST(registration_date AS TIMESTAMP)) AS earliest_registration,
        max(TRY_CAST(registration_date AS TIMESTAMP)) AS latest_registration,
        count(*) FILTER (WHERE TRY_CAST(registration_date AS TIMESTAMP) < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE TRY_CAST(registration_date AS TIMESTAMP) > TIMESTAMP '2026-06-17') AS after_window
    FROM bronze.customers
""")

In [ ]:
# Users note: check if casting type fails in silver
q("""
    SELECT
        typeof(registration_date),
        count(*) FILTER (WHERE registration_date is null)
    FROM silver.dim_customers
    GROUP BY ALL
""")

In [ ]:
# last_updated -- the spec docstring already flags a 2027 value; see exactly how many and how far out.
q("""
    SELECT
        max(TRY_CAST(last_updated AS TIMESTAMP)) AS latest_last_updated,
        count(*) FILTER (WHERE TRY_CAST(last_updated AS TIMESTAMP) > TIMESTAMP '2026-06-17') AS after_dataset_window,
        count(*) FILTER (WHERE TRY_CAST(last_updated AS TIMESTAMP) < TRY_CAST(registration_date AS TIMESTAMP)) AS updated_before_registered
    FROM bronze.customers
""")

In [ ]:
# What do the after-window last_updated rows actually look like?
q("""
    SELECT customer_id, registration_date, last_updated
    FROM bronze.customers
    WHERE TRY_CAST(last_updated AS TIMESTAMP) > TIMESTAMP '2026-06-17'
    ORDER BY last_updated DESC
    LIMIT 20
""")

## 7. Country canonicalization -- did `as_country()` actually resolve every spelling variant?

`COUNTRY_CANONICAL` in `config.py` maps known spelling splits (`mexico`/`méxico` -> `México`, etc.) to one canonical value. This checks whether *every* distinct raw value in `bronze.customers.country` is covered by that map -- an uncovered value falls through unchanged per `as_country()`'s own design, so it wouldn't error, it would just silently stay split in `silver.dim_customers.country`.

In [ ]:
print("Distinct raw country values in Bronze:")
display(q("SELECT country, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC"))

print("Distinct country values in Silver (post-canonicalization):")
display(q("SELECT country, count(*) AS n FROM silver.dim_customers GROUP BY 1 ORDER BY n DESC"))

In [ ]:
# The canonicalization map itself, mirrored from config.py -- keep in sync if that dict changes.
country_canonical = {
    "méxico": "México", "mexico": "México", "colombia": "Colombia", "argentina": "Argentina",
    "usa": "USA", "united states": "USA", "spain": "Spain", "brazil": "Brazil", "brasil": "Brazil",
}

raw_values = q("SELECT DISTINCT lower(trim(country)) AS v FROM bronze.customers WHERE country IS NOT NULL")["v"].tolist()
uncovered = [v for v in raw_values if v not in country_canonical]
print("Raw country values NOT covered by COUNTRY_CANONICAL (pass through unchanged):", uncovered or "none -- full coverage")

In [ ]:
# Users note: list of cities
q("SELECT len(city) as len, city, count(*) AS n FROM bronze.customers GROUP BY all ORDER BY len")

## 8. `postal_code` -- confirm the "kept as VARCHAR" reasoning

`table_specs.py`'s docstring says `postal_code` stays VARCHAR because only ~72.06% of values are numeric-parseable. Confirming that number directly against real data, since a stale profile claim (from synthetic dev data, not your production pull) could be wrong.

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE TRY_CAST(postal_code AS BIGINT) IS NOT NULL) AS numeric_parseable,
        round(100.0 * count(*) FILTER (WHERE TRY_CAST(postal_code AS BIGINT) IS NOT NULL) / count(*), 2) AS pct_numeric
    FROM bronze.customers
    WHERE postal_code IS NOT NULL AND trim(postal_code) <> ''
""")

## 9. `accepts_marketing` -- did boolean parsing behave as expected?

`as_boolean()` only recognizes literal `'true'`/`'false'` (case-insensitive); anything else becomes NULL rather than a guessed default. Comparing Bronze's raw distinct values against Silver's parsed boolean distribution shows exactly what got parsed vs. what became NULL and why.

In [ ]:
print("Bronze raw values:")
display(q("SELECT typeof(accepts_marketing), accepts_marketing, count(*) AS n FROM bronze.customers GROUP BY ALL ORDER BY n DESC"))

print("Silver parsed values:")
display(q("SELECT typeof(accepts_marketing), accepts_marketing, count(*) AS n FROM silver.dim_customers GROUP BY ALL ORDER BY n DESC"))

# Users note: added the typeof to check if the typecasting is correct 

## 10.5 Follow-up checks -- closing the gap with the `products` deep dive

Added after review: `document_number`'s UNIQUE constraint was never checked (only NOT NULL), and `email`/`mobile_phone`/`landline_phone`/`address`/`detected_accent`/`occupation` had no checks at all. `state` only had a NOT NULL check, no cross-check against `country`. These four cells close that gap.

In [ ]:
# document_number -- dictionary: NOT NULL, UNIQUE (same constraint products_spec's product_number
# has, which the products notebook checks directly -- this was missing here).
q("""
    SELECT
        count(*) AS total,
        count(DISTINCT document_number) AS distinct_document_number,
        count(*) FILTER (WHERE document_number IS NULL OR trim(document_number) = '') AS nulls
    FROM bronze.customers
""")

In [ ]:
# If any document_number repeats, is it the same customer_id each time (a resend/dup row, harmless)
# or two different customers sharing a document number (a real uniqueness violation)?
q("""
    SELECT document_number, count(*) AS n, count(DISTINCT customer_id) AS distinct_customer_ids
    FROM bronze.customers
    GROUP BY 1 HAVING count(*) > 1
    ORDER BY n DESC
    LIMIT 20
""")

In [ ]:
# email / mobile_phone / landline_phone / address -- no dictionary-declared format, so this is
# basic shape sanity rather than a pass/fail check: null rates, and how many populated values look
# structurally wrong (no '@' in email, non-digit-heavy phone numbers).
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE email IS NULL OR trim(email) = '') AS email_nulls,
        count(*) FILTER (WHERE email IS NOT NULL AND trim(email) <> '' AND email NOT LIKE '%@%') AS email_missing_at,
        count(*) FILTER (WHERE mobile_phone IS NULL OR trim(mobile_phone) = '') AS mobile_nulls,
        count(*) FILTER (WHERE mobile_phone IS NOT NULL AND trim(mobile_phone) <> ''
                          AND regexp_matches(mobile_phone, '[0-9]')  = false) AS mobile_no_digits,
        count(*) FILTER (WHERE landline_phone IS NULL OR trim(landline_phone) = '') AS landline_nulls,
        count(*) FILTER (WHERE address IS NULL OR trim(address) = '') AS address_nulls
    FROM bronze.customers
""")

In [ ]:
# detected_accent / occupation -- no declared domain either; just seeing the actual value spread
# and null rates, same treatment as marital_status/education_level got originally.
print("detected_accent:")
display(q("SELECT detected_accent, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC"))

print("occupation (top 20 by frequency -- free text, so this is a spot-check, not a full domain):")
display(q("SELECT occupation, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC LIMIT 20"))

print("occupation null rate:")
display(q("SELECT count(*) AS total, count(*) FILTER (WHERE occupation IS NULL OR trim(occupation) = '') AS nulls FROM bronze.customers"))

In [ ]:
# state vs. country -- no declared FK/lookup between them, so this can't be a strict pass/fail,
# but a state value that never co-occurs with a plausible country would be worth a closer look.
# Shows the state/country pairs and their row counts so you can eyeball anything that looks wrong.
q("""
    SELECT country, state, count(*) AS n
    FROM bronze.customers
    GROUP BY 1, 2
    ORDER BY country, n DESC
""")

## 10. Known issues already confirmed elsewhere (recap, not re-derived here)

- **`registration_branch_id` -> `branches.branch_id`**: 99.997% orphaned (149,995/150,000). Confirmed a real source-data defect, not a Silver bug or format mismatch -- see `diagnose_branch_fk.ipynb` and the note added to `data_pipelines/silver/README.md`.
- **`customer_status` domain violation**: 4,407/150,000 rows (~2.94%) outside `{Active, Inactive, Blocked, Closed}` -- see section 4 above for exactly which values those are.

## Checklist -- fill in after running the sections above

- [ ] Column inventory (section 1): spec and table agree, no drift
- [ ] Row reconciliation (section 2): Silver row count == Bronze distinct customer_id count
- [ ] NOT NULL audit (section 3): note any column the dictionary claims is NOT NULL but isn't in practice
- [ ] Domain checks (section 4): note any category value outside the declared set, beyond the known `customer_status` warning
- [ ] Numeric ranges (section 5): note any `credit_score` outside 300-850 or negative income
- [ ] Date sanity (section 6): note implausible ages, out-of-window registrations, or `last_updated` anomalies beyond the one already documented
- [ ] Country canonicalization (section 7): confirm `uncovered` list is empty
- [ ] `postal_code` (section 8): confirm the ~72% numeric-parseable claim still holds
- [ ] `accepts_marketing` (section 9): confirm the NULL rate after parsing is explained by genuinely unparseable source values, not a transform bug

## Your own exploration

Free cells below -- `con`, `q()`, and `q_full()` are already set up.

### Users note: Duplicated first and last names, how can we treat them?

In [ ]:
q("SELECT len(first_name) as len, min(first_name), max(first_name), count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY len")

In [ ]:
q("SELECT * FROM bronze.customers WHERE first_name in ('Óscar Óscar', 'Adriana Adriana', 'Victoria Victoria', 'Valentina Valentina', 'Alejandra Alejandra')")

In [ ]:
q("SELECT len(last_name) as len, min(last_name), max(last_name), count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY len")

In [ ]:
q("SELECT * FROM bronze.customers WHERE last_name in ('Cruz Cruz', 'Vera Vera', 'Acosta Acosta', 'Álvarez Álvarez', 'Rodríguez Rodríguez', 'Contreras Contreras')")

In [ ]:
# Users note: i want to do this type of side by side comparisson of bronze and silver
print("Bronze raw values:")
display(q("SELECT typeof(customer_status) , len(customer_status) as len, customer_status, count(*) AS n FROM bronze.customers GROUP BY all ORDER BY customer_status"))

print("Silver parsed values:")
display(q("SELECT typeof(customer_status) , len(customer_status) as len, customer_status, count(*) AS n FROM silver.dim_customers GROUP BY all ORDER BY customer_status"))

In [ ]:
con.close()
print("OK")